# Handwritten Character and Digit Recognition (0–9 + A–Z)

**Goal:** train one CNN to recognize 36 classes: digits `0–9` and uppercase English letters `A–Z`.

We combine:
- **MNIST** for handwritten digits.
- **EMNIST Letters** for handwritten uppercase letters.

Run the cells from top to bottom. In Colab, select **Runtime → Change runtime type → T4 GPU** if available. Free GPU access is not guaranteed; CPU execution is also possible but slower.

**Label mapping used:** `0–9` map to class indices `0–9`; EMNIST letter labels `1–26` map to class indices `10–35` (A–Z).

## 1. Install and import libraries
TensorFlow Datasets downloads the public datasets automatically.

In [ ]:
!pip -q install tensorflow-datasets

import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds

print("TensorFlow:", tf.__version__)
print("Num GPUs Available:", len(tf.config.list_physical_devices("GPU")))

## 2. Load MNIST and EMNIST Letters
The first download may take a few minutes. `as_supervised=True` returns `(image, label)` pairs.

In [ ]:
(ds_mnist_train, ds_mnist_test), mnist_info = tfds.load(
    "mnist",
    split=["train", "test"],
    as_supervised=True,
    with_info=True
)

(ds_letters_train, ds_letters_test), letters_info = tfds.load(
    "emnist/letters",
    split=["train", "test"],
    as_supervised=True,
    with_info=True
)

print("MNIST train examples:", mnist_info.splits["train"].num_examples)
print("EMNIST Letters train examples:", letters_info.splits["train"].num_examples)

## 3. Prepare and combine datasets

- Convert pixel values to `float32` in the range 0–1.
- Add a channel dimension so each image has shape `28 × 28 × 1`.
- Keep digit labels as 0–9.
- Shift EMNIST letter labels from 1–26 to 10–35.

This notebook uses a manageable subset of EMNIST training examples to keep the free Colab run practical. Increase the limit later if you want.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
BATCH_SIZE = 128
LETTER_TRAIN_LIMIT = 80000  # set to None to use the full EMNIST Letters training split

def prepare_digit(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.reshape(image, (28, 28, 1))
    label = tf.cast(label, tf.int64)
    return image, label

def prepare_letter(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.reshape(image, (28, 28, 1))
    # EMNIST Letters labels are 1..26. Map them to 10..35.
    label = tf.cast(label, tf.int64) + 9
    return image, label

digit_train = ds_mnist_train.map(prepare_digit, num_parallel_calls=AUTOTUNE)
digit_test = ds_mnist_test.map(prepare_digit, num_parallel_calls=AUTOTUNE)

letter_train = ds_letters_train.map(prepare_letter, num_parallel_calls=AUTOTUNE)
letter_test = ds_letters_test.map(prepare_letter, num_parallel_calls=AUTOTUNE)

if LETTER_TRAIN_LIMIT is not None:
    letter_train = letter_train.take(LETTER_TRAIN_LIMIT)

train_ds = digit_train.concatenate(letter_train)
test_ds = digit_test.concatenate(letter_test)

train_ds = train_ds.shuffle(30000, reshuffle_each_iteration=True).batch(BATCH_SIZE).prefetch(AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

print("Datasets are ready.")

## 4. Display sample images
Check that the letters are upright and readable. If your downloaded EMNIST samples appear rotated, tell me and we will adjust preprocessing before training.

In [ ]:
class_names = [str(i) for i in range(10)] + [chr(ord("A") + i) for i in range(26)]

for images, labels in train_ds.take(1):
    plt.figure(figsize=(10, 5))
    for i in range(15):
        plt.subplot(3, 5, i + 1)
        plt.imshow(images[i].numpy().squeeze(), cmap="gray")
        plt.title(class_names[int(labels[i])])
        plt.axis("off")
    plt.tight_layout()
    plt.show()

## 5. Build the CNN model
CNN layers learn visual patterns such as edges, curves and strokes. The final Dense layer has 36 outputs, one for each class.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28, 1)),
    tf.keras.layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.30),
    tf.keras.layers.Dense(36, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## 6. Train the model
Start with 5 epochs to confirm everything works. You can increase to 10–15 epochs after the first successful run.

In [ ]:
EPOCHS = 5

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS
)

## 7. Plot accuracy and loss

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Training accuracy")
plt.plot(history.history["val_accuracy"], label="Validation accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Model Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Training loss")
plt.plot(history.history["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Model Loss")
plt.legend()

plt.tight_layout()
plt.show()

## 8. Evaluate the model
Record the actual test accuracy shown by your run. Do not write a guessed accuracy in your report.

In [ ]:
test_loss, test_accuracy = model.evaluate(test_ds)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

## 9. Test a few predictions

In [ ]:
for images, labels in test_ds.take(1):
    predictions = model.predict(images[:12], verbose=0)
    predicted_ids = np.argmax(predictions, axis=1)

    plt.figure(figsize=(12, 6))
    for i in range(12):
        plt.subplot(3, 4, i + 1)
        plt.imshow(images[i].numpy().squeeze(), cmap="gray")
        actual = class_names[int(labels[i])]
        predicted = class_names[int(predicted_ids[i])]
        plt.title(f"Actual: {actual}\nPredicted: {predicted}")
        plt.axis("off")
    plt.tight_layout()
    plt.show()

## 10. Save the trained model
Download this file from Colab's Files panel and copy it to the project's `models` folder in VS Code.

In [ ]:
os.makedirs("models", exist_ok=True)
model.save("handwritten_cnn.keras")
print("Saved model:", os.path.abspath("handwritten_cnn.keras"))

from google.colab import files
files.download("handwritten_cnn.keras")

## 11. Run the Streamlit app in VS Code

1. Copy the downloaded `handwritten_cnn.keras` file into the project's `models/` folder.
2. Open the extracted project folder in VS Code.
3. Create and activate a virtual environment.
4. Install requirements and run:

```bash
pip install -r requirements.txt
streamlit run app.py
```

The app accepts an image containing one handwritten character and shows the predicted class and top-5 confidence scores.

## 12. Conclusion (for your practical/project report)

A CNN-based handwritten recognition model was developed to classify 36 categories consisting of digits 0–9 and uppercase letters A–Z. The model was trained using MNIST and EMNIST Letters datasets. Its actual test accuracy and loss should be filled in from the evaluation output after running the notebook.